# LeWorldModelRec — Training on Colab T4

Everything runs **locally on the Colab instance** (no Drive mount).

**Pipeline :**
1. Check GPU
2. Clone repo + patch losses
3. Generate simple pendulum dataset
4. Train LeWorldModelRec (reconstruction + perceptual + freq + SIGReg)
5. Plot loss curves
6. Visualise reconstructions
7. Download best checkpoint

**Différence vs LeWorldModel (JEPA) :**  
Pas de target encoder EMA — supervision directe en pixel space via MSE + perceptual loss (VGG16) + frequency loss (FFT).

> Runtime → Change runtime type → **T4 GPU** before running.

## 1. GPU check

In [ ]:
import subprocess, torch

print(subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader'))
print(f'PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}  |  device: {torch.cuda.get_device_name(0)}')
assert torch.cuda.is_available(), 'No GPU found — switch to T4 runtime first.'

## 2. Clone repo

In [ ]:
import os

REPO_URL = 'https://github.com/JulesV19/double-pendule-wolrdmodel.git'
REPO_DIR = '/content/WorldModel'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

### 2b. Patch models — écrase les fichiers locaux avec la version correcte

Applique les fixes qui ne sont pas encore pushés sur GitHub :
- `losses.py` : sous-échantillonnage avant VGG/FFT (`max_samples`) pour éviter l'OOM
- `__init__.py` : suppression des imports `ae` et `rssm` supprimés du repo

In [ ]:
from pathlib import Path

Path('models/__init__.py').write_text(
"""from .encoder  import ContextEncoder
from .decoder  import Decoder
from .sigreg   import sigreg_loss
from .losses   import PerceptualLoss, FrequencyLoss
from .jepa.model import LeWorldModel, TransitionPredictor
from .rec.model  import LeWorldModelRec
"""
)
print('models/__init__.py patched.')

In [ ]:
Path('models/losses.py').write_text(
r"""
import torch
import torch.nn as nn
import torch.nn.functional as F


def _subsample(pred, target, max_samples):
    C, H, W = pred.shape[-3], pred.shape[-2], pred.shape[-1]
    p = pred.reshape(-1, C, H, W)
    t = target.reshape(-1, C, H, W).detach()
    n = p.shape[0]
    if n > max_samples:
        idx = torch.randperm(n, device=p.device)[:max_samples]
        p = p[idx]
        t = t[idx]
    return p, t


class PerceptualLoss(nn.Module):
    """
    Perceptual loss VGG16 avec sous-echantillonnage.
    max_samples : nombre max de frames par appel VGG (evite l'OOM sur T4).
    """

    _SLICES = [(0, 4), (4, 9), (9, 16)]

    def __init__(self, weights=(1.0, 1.0, 1.0), max_samples=64):
        super().__init__()
        self.max_samples = max_samples
        from torchvision import models
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        self.slices = nn.ModuleList([
            nn.Sequential(*list(vgg.features.children())[a:b])
            for a, b in self._SLICES
        ])
        self.weights = weights
        for p in self.parameters():
            p.requires_grad_(False)
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, pred, target):
        pred, target = _subsample(pred, target, self.max_samples)
        xp = (pred   - self.mean) / self.std
        xt = (target - self.mean) / self.std
        loss = torch.tensor(0.0, device=pred.device, dtype=pred.dtype)
        for w, slice_ in zip(self.weights, self.slices):
            xp = slice_(xp)
            xt = slice_(xt)
            loss = loss + w * F.mse_loss(xp, xt)
        return loss


class FrequencyLoss(nn.Module):
    """Perte FFT 2D avec sous-echantillonnage."""

    def __init__(self, high_freq_boost=True, max_samples=256):
        super().__init__()
        self.high_freq_boost = high_freq_boost
        self.max_samples     = max_samples

    @staticmethod
    def _freq_weight(h, w, device):
        fy = torch.fft.fftfreq(h, device=device).abs()
        fx = torch.fft.rfftfreq(w, device=device).abs()
        return (fy[:, None] + fx[None, :]).clamp(min=1e-6)

    def forward(self, pred, target):
        pred, target = _subsample(pred, target, self.max_samples)
        pred_f   = torch.fft.rfft2(pred,   norm='ortho')
        target_f = torch.fft.rfft2(target, norm='ortho')
        if self.high_freq_boost:
            w = self._freq_weight(pred.shape[-2], pred.shape[-1], pred.device)
            pred_f   = pred_f   * w
            target_f = target_f * w
        return (F.mse_loss(pred_f.real, target_f.real)
              + F.mse_loss(pred_f.imag, target_f.imag))
"""
)
print('models/losses.py patched.')

## 3. Install dependencies

In [ ]:
# torchvision is required for PerceptualLoss (VGG16)
!pip install -q Pillow matplotlib numpy torchvision
print('Dependencies ready.')

## 4. Generate dataset

Generates **2 000 trajectories × 500 frames** (64×64 px) of a simple pendulum.  
Each trajectory has random initial angle **and** random initial velocity.  
~5-10 min on Colab CPU.

During training, a random **window of 32 frames** is sampled from each 500-frame
trajectory at every epoch.

In [ ]:
from pathlib import Path

DATASET_DIR = 'dataset/pendulum'
N_TRAJ      = 2000
N_FRAMES    = 500
IMG_SIZE    = 64

existing = len(list(Path(DATASET_DIR).glob('traj_*.npz'))) if Path(DATASET_DIR).exists() else 0

if existing >= N_TRAJ:
    print(f'Dataset already present ({existing} trajectories) — skipping generation.')
else:
    print(f'Generating {N_TRAJ} trajectories × {N_FRAMES} frames…')
    from generate_dataset import generate_dataset
    generate_dataset(
        n_trajectories=N_TRAJ,
        n_frames=N_FRAMES,
        img_size=IMG_SIZE,
        dt=0.05,
        output_dir=DATASET_DIR,
        seed=42,
    )

### Quick dataset preview

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sample = np.load(f'{DATASET_DIR}/traj_0000.npz')
frames = sample['frames']   # (T, H, W, 3)

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
fig.patch.set_facecolor('#111')
for ax, idx in zip(axes, np.linspace(0, len(frames)-1, 8, dtype=int)):
    ax.imshow(frames[idx])
    ax.axis('off')
    ax.set_title(f't={idx}', color='white', fontsize=8)
plt.suptitle('Trajectory 0 — 8 frames', color='white')
plt.tight_layout()
plt.show()

## 5. Training

| Hyperparameter | Value | Note |
|---|---|---|
| `embed_dim` | 128 | latent dimension |
| `hidden_dim` | 512 | MLP transition FFN |
| `lam` | 0.5 | SIGReg weight — **aligné avec JEPA** |
| `rec_coef` | 1.0 | poids MSE reconstruction |
| `pred_coef` | 1.0 | poids MSE prédiction rollout |
| `perceptual_coef` | 0.1 | poids perceptual loss VGG16 — max 64 frames/appel |
| `freq_coef` | 0.05 | poids frequency loss FFT — max 256 frames/appel |
| `rollout_k` | 10 | horizon de prédiction — **aligné avec JEPA** (0.5 s = 10 × dt) |
| `seq_len` | 32 | réduit vs JEPA — chaque step rollout passe par le décodeur (OOM sinon) |
| `batch_size` | 16 | réduit — B×T frames stockées en graph pour backprop |
| `epochs` | 50 | |
| `lr` | 1e-4 | AdamW + warmup 5ep + cosine |

**Asymétrie résiduelle documentée :** `seq_len=32` (AE) vs `seq_len=100` (JEPA) — contrainte mémoire due au décodeur pixel-space. Non corrigeable sans réduire la qualité de reconstruction.

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────────
EMBED_DIM        = 128
HIDDEN_DIM       = 512
LAM              = 0.5    # SIGReg weight — aligné avec JEPA
REC_COEF         = 1.0    # poids reconstruction MSE
PRED_COEF        = 1.0    # poids prédiction rollout MSE
PERCEPTUAL_COEF  = 0.1    # poids perceptual loss VGG16 (0 = désactivé)
FREQ_COEF        = 0.05   # poids frequency loss FFT (0 = désactivé)
ROLLOUT_K        = 10     # horizon de prédiction — aligné avec JEPA (0.5 s = 10 × dt)
SEQ_LEN          = 32     # fenêtre tirée aléatoirement — réduit pour éviter OOM décodeur
N_PROJ           = 512
EPOCHS           = 50
BATCH_SIZE       = 16     # réduit — B*T frames stockées en graph
LR               = 1e-4
WEIGHT_DECAY     = 0.05
NUM_WORKERS      = 2
CKPT_DIR         = 'checkpoints'
VIS_DIR          = 'visuals'
RESUME           = None   # set to 'checkpoints/rec/lewm_rec_best.pt' to resume

In [ ]:
import time, math
from pathlib import Path

import torch
import torch.optim as optim
import matplotlib.pyplot as plt

from models.rec.model import LeWorldModelRec
from data.dataset import make_seq_dataloaders

device = torch.device('cuda')
print(f'Training on: {torch.cuda.get_device_name(0)}')

# ── Dataloaders ────────────────────────────────────────────────────────────────
train_loader, val_loader = make_seq_dataloaders(
    dataset_dir=DATASET_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    seq_len=SEQ_LEN,
)
print(f'Train: {len(train_loader)} batches  |  Val: {len(val_loader)} batches')

# ── Model ──────────────────────────────────────────────────────────────────────
model = LeWorldModelRec(
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    lam=LAM,
    rec_coef=REC_COEF,
    pred_coef=PRED_COEF,
    perceptual_coef=PERCEPTUAL_COEF,
    freq_coef=FREQ_COEF,
    n_proj=N_PROJ,
    rollout_k=ROLLOUT_K,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_params:,}')

# ── Optimizer & scheduler ──────────────────────────────────────────────────────
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch):
    warmup = 5
    if epoch < warmup:
        return (epoch + 1) / warmup
    t = (epoch - warmup) / max(1, EPOCHS - warmup)
    return 0.5 * (1 + math.cos(math.pi * t))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ── Evaluate ───────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    keys = ['loss', 'rec_loss', 'pred_loss', 'perc_loss', 'freq_loss', 'sigreg']
    sums = {k: 0.0 for k in keys}
    for frames, _ in loader:
        m = model(frames.to(device, non_blocking=True))
        for k in sums:
            sums[k] += m[k].item()
    n = len(loader)
    return {k: v / n for k, v in sums.items()}

# ── Resume ─────────────────────────────────────────────────────────────────────
start_epoch = 1
if RESUME and Path(RESUME).exists():
    ckpt = torch.load(RESUME, map_location=device)
    model.load_state_dict(ckpt['model'], strict=False)
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch = ckpt['epoch'] + 1
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda, last_epoch=ckpt['epoch'])
    print(f'Resumed from epoch {ckpt["epoch"]}')

Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)
Path(VIS_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:

# ── Monitoring : puissance GPU · mémoire · temps ───────────────────────────────
import threading, subprocess

_power_readings = []
_stop_monitor   = threading.Event()

def _monitor_power():
    while not _stop_monitor.is_set():
        try:
            out = subprocess.check_output(
                ['nvidia-smi', '--query-gpu=power.draw', '--format=csv,noheader,nounits'],
                text=True
            ).strip()
            _power_readings.append(float(out))
        except Exception:
            pass
        time.sleep(2)

torch.cuda.reset_peak_memory_stats()
_monitor_thread    = threading.Thread(target=_monitor_power, daemon=True)
_monitor_thread.start()
_train_start_wall  = time.time()
print("Monitoring démarré : puissance GPU (toutes les 2s), mémoire peak, temps total.")


In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────────

history             = {k: [] for k in ['train', 'val', 'rec', 'pred', 'perc', 'freq', 'sigreg']}
best_val            = float('inf')
best_epoch          = 1
time_per_epoch      = []
memory_per_epoch_MB = []

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    t0   = time.time()
    keys = ['loss', 'rec_loss', 'pred_loss', 'perc_loss', 'freq_loss', 'sigreg']
    sums = {k: 0.0 for k in keys}

    for frames, _ in train_loader:
        frames = frames.to(device, non_blocking=True)
        optimizer.zero_grad()
        m = model(frames)
        m['loss'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        for k in sums:
            sums[k] += m[k].item()

    scheduler.step()
    n          = len(train_loader)
    train_loss = sums['loss'] / n
    val_m      = evaluate(model, val_loader)

    history['train'].append(train_loss)
    history['val'].append(val_m['loss'])
    history['rec'].append(sums['rec_loss'] / n)
    history['pred'].append(sums['pred_loss'] / n)
    history['perc'].append(sums['perc_loss'] / n)
    history['freq'].append(sums['freq_loss'] / n)
    history['sigreg'].append(sums['sigreg'] / n)

    elapsed = time.time() - t0
    time_per_epoch.append(round(elapsed, 2))
    memory_per_epoch_MB.append(round(torch.cuda.max_memory_allocated() / 1e6, 1))

    improved = val_m['loss'] < best_val
    if improved:
        best_val   = val_m['loss']
        best_epoch = epoch
        torch.save({
            'epoch':     epoch,
            'model':     model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'val_loss':  best_val,
            'args': {
                'embed_dim':       EMBED_DIM,
                'hidden_dim':      HIDDEN_DIM,
                'lam':             LAM,
                'rec_coef':        REC_COEF,
                'pred_coef':       PRED_COEF,
                'perceptual_coef': PERCEPTUAL_COEF,
                'freq_coef':       FREQ_COEF,
                'n_proj':          N_PROJ,
                'rollout_k':       ROLLOUT_K,
            },
        }, f'{CKPT_DIR}/lewm_rec_best.pt')

    lr_now = optimizer.param_groups[0]['lr']
    print(
        f'Epoch {epoch:3d}/{EPOCHS}'
        f'  loss={train_loss:.4f}'
        f'  rec={sums["rec_loss"]/n:.4f}'
        f'  pred={sums["pred_loss"]/n:.4f}'
        f'  perc={sums["perc_loss"]/n:.4f}'
        f'  freq={sums["freq_loss"]/n:.4f}'
        f'  sig={sums["sigreg"]/n:.4f}'
        f'  val={val_m["loss"]:.4f}'
        f'  lr={lr_now:.2e}'
        f'  mem={memory_per_epoch_MB[-1]:.0f}MB'
        f'  {elapsed:.1f}s'
        + ('  <-- best' if improved else '')
    )

print(f'\nBest val loss: {best_val:.4f}  @epoch {best_epoch}  ->  {CKPT_DIR}/lewm_rec_best.pt')

## 6. Loss curves

In [ ]:
epochs_x = range(1, len(history['train']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.patch.set_facecolor('#111')

ax = axes[0]
ax.set_facecolor('#111')
ax.plot(epochs_x, history['train'], color='#4fc3f7', label='train')
ax.plot(epochs_x, history['val'],   color='#ff8a65', label='val')
ax.set_title('Total loss', color='white')
ax.set_xlabel('epoch', color='white')
ax.legend(facecolor='#222', labelcolor='white', edgecolor='#444')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')

ax = axes[1]
ax.set_facecolor('#111')
ax.plot(epochs_x, history['rec'],  color='#a5d6a7', label='rec MSE')
ax.plot(epochs_x, history['pred'], color='#4fc3f7', label='pred MSE')
ax.set_title('Pixel losses (MSE)', color='white')
ax.set_xlabel('epoch', color='white')
ax.legend(facecolor='#222', labelcolor='white', edgecolor='#444')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')

ax = axes[2]
ax.set_facecolor('#111')
ax.plot(epochs_x, history['perc'],   color='#ffcc80', label='perceptual (VGG)')
ax.plot(epochs_x, history['freq'],   color='#ef9a9a', label='frequency (FFT)')
ax.plot(epochs_x, history['sigreg'], color='#ce93d8', label='SIGReg')
ax.set_title('Anti-blur + Regularization', color='white')
ax.set_xlabel('epoch', color='white')
ax.legend(facecolor='#222', labelcolor='white', edgecolor='#444')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')

plt.tight_layout()
plt.savefig(f'{VIS_DIR}/lewm_rec_loss_colab.png', dpi=100, bbox_inches='tight', facecolor='#111')
plt.show()

## 7. Reconstruction visualisation

Avantage clé de l'architecture AE : on peut décoder les latents et voir ce que le modèle a appris.  
- **Ligne 1** : frames originales  
- **Ligne 2** : reconstructions `decode(encode(frame))`  
- **Ligne 3** : prédictions `decode(predictor(encode(frame_t))) ≈ frame_{t+1}`

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

ckpt = torch.load(f'{CKPT_DIR}/lewm_rec_best.pt', map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Loaded checkpoint from epoch {ckpt["epoch"]}  (val_loss={ckpt["val_loss"]:.4f})')

frames_batch, _ = next(iter(val_loader))
sample  = frames_batch[:1].to(device)            # (1, T, 3, H, W)
T_SHOW  = 8
indices = np.linspace(0, sample.shape[1] - 2, T_SHOW, dtype=int)

with torch.no_grad():
    z        = model.encode(sample)              # (1, T, D)
    rec      = model.decode(z)                   # (1, T, 3, H, W)
    z_pred   = model.predictor(z[:, :-1])        # (1, T-1, D)
    pred_dec = model.decode(z_pred)              # (1, T-1, 3, H, W)

orig_np  = sample[0, indices].permute(0, 2, 3, 1).clamp(0, 1).cpu().numpy()
rec_np   = rec[0, indices].permute(0, 2, 3, 1).clamp(0, 1).cpu().numpy()
pred_np  = pred_dec[0, indices].permute(0, 2, 3, 1).clamp(0, 1).cpu().numpy()

fig, axes = plt.subplots(3, T_SHOW, figsize=(T_SHOW * 2, 7))
fig.patch.set_facecolor('#111')
row_labels = ['Original', 'Reconstruction', 'Prediction (t+1)']
row_data   = [orig_np, rec_np, pred_np]

for row, (label, data) in enumerate(zip(row_labels, row_data)):
    for col in range(T_SHOW):
        ax = axes[row, col]
        ax.imshow(data[col])
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(label, color='white', fontsize=10, labelpad=6)
        if row == 0:
            ax.set_title(f't={indices[col]}', color='white', fontsize=8)

plt.suptitle('LeWorldModelRec — reconstructions & prédictions', color='white', fontsize=12)
plt.tight_layout()
plt.savefig(f'{VIS_DIR}/lewm_rec_visuals.png', dpi=120, bbox_inches='tight', facecolor='#111')
plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Section 8 — Post-training analysis                                          ║
# ║  Énergie · Mémoire · Temps · Probe linéaire + MLP · Rollout pixel + latent   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
import json
from datetime import datetime
from data.dataset import PendulumSeqDataset
from torch.utils.data import random_split
import torch.nn.functional as F

# ── Arrêt monitor + totaux ─────────────────────────────────────────────────────
_stop_monitor.set()
_total_time_s = time.time() - _train_start_wall
avg_power_W   = float(np.mean(_power_readings)) if _power_readings else 0.0
energy_Wh     = avg_power_W * _total_time_s / 3600
peak_mem_MB   = torch.cuda.max_memory_allocated() / 1e6
n_params      = sum(p.numel() for p in model.parameters())
n_grad_steps  = len(history['train']) * len(train_loader)

print(f"Temps total    : {_total_time_s/60:.1f} min  ({n_grad_steps:,} gradient steps)")
print(f"Énergie GPU    : {avg_power_W:.1f} W moy  →  {energy_Wh:.3f} Wh  ({avg_power_W*_total_time_s:.0f} J)")
print(f"Mémoire pic    : {peak_mem_MB:.0f} MB  |  Paramètres : {n_params/1e6:.3f} M")

# ── Reload best checkpoint ─────────────────────────────────────────────────────
ckpt_best = torch.load(f'{CKPT_DIR}/lewm_rec_best.pt', map_location=device)
model.load_state_dict(ckpt_best['model'])
model.eval()

# ── Helpers ────────────────────────────────────────────────────────────────────
def _extract_z(mdl, indices, ds_dir, dev, chunk=16):
    full_ds = PendulumSeqDataset(ds_dir, seq_len=None)
    Zs, Ls  = [], []
    for idx in indices:
        frames, states = full_ds[idx]
        T  = frames.shape[0]
        zc = []
        for s in range(0, T, chunk):
            f = frames[s:s+chunk].unsqueeze(0).to(dev)
            with torch.no_grad():
                zc.append(mdl.encode(f)[0].cpu())
        z_all = torch.cat(zc, 0)
        Zs.append(z_all[1:])
        Ls.append(states[1:])
    return torch.cat(Zs).float(), torch.cat(Ls).float()

def _r2(pred, true):
    ss_res = ((true - pred)**2).sum(0)
    ss_tot = ((true - true.mean(0))**2).sum(0)
    r = 1 - ss_res / ss_tot.clamp(min=1e-8)
    return {'r2_theta': round(r[0].item(), 4), 'r2_omega': round(r[1].item(), 4),
            'r2_mean': round(r.mean().item(), 4)}

def _run_probe(Z_tr, L_tr, Z_val, L_val, epochs=150, hidden=256):
    import torch.nn as nn, torch.utils.data as td
    zm, zs = Z_tr.mean(0), Z_tr.std(0).clamp(min=1e-6)
    Zn_tr  = (Z_tr - zm) / zs
    Zn_val = (Z_val - zm) / zs
    D, out = Zn_tr.shape[1], {}
    for name, net, lr in [
        ('linear', nn.Linear(D, 2), 1e-2),
        ('mlp',    nn.Sequential(nn.Linear(D, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, 2)), 1e-3),
    ]:
        opt = torch.optim.Adam(net.parameters(), lr=lr)
        dl  = td.DataLoader(td.TensorDataset(Zn_tr, L_tr), batch_size=1024, shuffle=True)
        for _ in range(epochs):
            net.train()
            for zb, lb in dl:
                loss = F.mse_loss(net(zb), lb)
                opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            out[name] = _r2(net(Zn_val), L_val)
        print(f"  probe_{name:6s}  R²(θ)={out[name]['r2_theta']:.4f}  "
              f"R²(ω)={out[name]['r2_omega']:.4f}  R²(mean)={out[name]['r2_mean']:.4f}")
    return out

@torch.no_grad()
def _rollout_latent(mdl, val_idx, ds_dir, dev, steps=(1, 5, 10, 20), n_traj=30):
    full_ds = PendulumSeqDataset(ds_dir, seq_len=None)
    cos_acc = {k: [] for k in steps}
    mse_acc = {k: [] for k in steps}
    max_k   = max(steps)
    for idx in list(val_idx)[:n_traj]:
        frames, _ = full_ds[idx]
        T = frames.shape[0]
        if T < max_k + 5: continue
        z = mdl.encode(frames.unsqueeze(0).to(dev))[0]
        for t in range(1, min(T - max_k, 50)):
            z_r = z[t:t+1]
            for k in range(1, max_k + 1):
                z_r = mdl.predictor(z_r)
                if k in steps:
                    z_true = z[t+k:t+k+1]
                    cos_acc[k].append(F.cosine_similarity(z_r, z_true).item())
                    mse_acc[k].append(F.mse_loss(z_r, z_true).item())
    return {
        **{f'cosine_sim_step{k}': round(float(np.mean(cos_acc[k])), 4)
           for k in steps if cos_acc[k]},
        **{f'latent_mse_step{k}': round(float(np.mean(mse_acc[k])), 6)
           for k in steps if mse_acc[k]},
    }

@torch.no_grad()
def _rollout_pixel(mdl, val_idx, ds_dir, dev, steps=(1, 5, 10), n_traj=20):
    """MSE pixel entre frame prédite décodée et frame réelle — métrique propre à l'AE."""
    full_ds = PendulumSeqDataset(ds_dir, seq_len=None)
    mse_acc = {k: [] for k in steps}
    max_k   = max(steps)
    for idx in list(val_idx)[:n_traj]:
        frames, _ = full_ds[idx]
        T = frames.shape[0]
        if T < max_k + 5: continue
        f = frames.unsqueeze(0).to(dev)
        z = mdl.encode(f)[0]                          # (T, D)
        for t in range(1, min(T - max_k, 30)):
            z_r = z[t:t+1]
            for k in range(1, max_k + 1):
                z_r = mdl.predictor(z_r)
                if k in steps:
                    f_pred = mdl.decode(z_r.unsqueeze(1))[0, 0]   # (3, H, W)
                    f_true = frames[t+k].to(dev)
                    mse_acc[k].append(F.mse_loss(f_pred, f_true).item())
    return {f'pixel_mse_step{k}': round(float(np.mean(mse_acc[k])), 6)
            for k in steps if mse_acc[k]}

# ── Run ────────────────────────────────────────────────────────────────────────
print('\n[1/3] Probe linéaire + MLP…')
full_ds = PendulumSeqDataset(DATASET_DIR, seq_len=None)
n_ds    = len(full_ds)
gen     = torch.Generator().manual_seed(42)
tr_idx, val_idx = random_split(range(n_ds), [int(n_ds*0.9), n_ds - int(n_ds*0.9)], generator=gen)
Z_tr, L_tr   = _extract_z(model, list(tr_idx),  DATASET_DIR, device)
Z_val, L_val = _extract_z(model, list(val_idx), DATASET_DIR, device)
probe_res    = _run_probe(Z_tr, L_tr, Z_val, L_val)

print('\n[2/3] Rollout quality latent…')
rollout_lat = _rollout_latent(model, list(val_idx), DATASET_DIR, device)
for k, v in rollout_lat.items(): print(f'  {k}: {v}')

print('\n[3/3] Rollout quality pixel…')
rollout_pix = _rollout_pixel(model, list(val_idx), DATASET_DIR, device)
for k, v in rollout_pix.items(): print(f'  {k}: {v}')

# ── Save JSON ──────────────────────────────────────────────────────────────────
stats = {
    'model':     'rec',
    'timestamp': datetime.now().isoformat(),
    'hyperparams': {
        'embed_dim': EMBED_DIM, 'hidden_dim': HIDDEN_DIM,
        'lam': LAM, 'rollout_k': ROLLOUT_K,
        'rec_coef': REC_COEF, 'pred_coef': PRED_COEF,
        'perceptual_coef': PERCEPTUAL_COEF, 'freq_coef': FREQ_COEF,
        'n_proj': N_PROJ, 'seq_len': SEQ_LEN,
        'batch_size': BATCH_SIZE, 'epochs': EPOCHS,
        'lr': LR, 'weight_decay': WEIGHT_DECAY,
    },
    'time': {
        'total_s':         round(_total_time_s, 1),
        'total_min':       round(_total_time_s / 60, 2),
        'per_epoch_s':     time_per_epoch,
        'avg_per_epoch_s': round(float(np.mean(time_per_epoch)), 2),
        'gradient_steps':  n_grad_steps,
        'steps_per_sec':   round(n_grad_steps / _total_time_s, 2),
    },
    'memory': {
        'peak_gpu_MB':       round(peak_mem_MB, 1),
        'model_params_M':    round(n_params / 1e6, 3),
        'trainable_params':  n_params,
        'per_epoch_peak_MB': memory_per_epoch_MB,
        'max_per_epoch_MB':  max(memory_per_epoch_MB),
    },
    'energy': {
        'avg_power_W':      round(avg_power_W, 2),
        'total_Wh':         round(energy_Wh, 4),
        'total_J':          round(avg_power_W * _total_time_s, 1),
        'n_power_samples':  len(_power_readings),
        'power_readings_W': [round(p, 1) for p in _power_readings],
    },
    'convergence': {
        'best_val_loss':  round(best_val, 6),
        'best_epoch':     best_epoch,
        'epochs_trained': len(history['train']),
        'train_loss':     [round(x, 6) for x in history['train']],
        'val_loss':       [round(x, 6) for x in history['val']],
        'rec_loss':       [round(x, 6) for x in history['rec']],
        'pred_loss':      [round(x, 6) for x in history['pred']],
        'perc_loss':      [round(x, 6) for x in history['perc']],
        'freq_loss':      [round(x, 6) for x in history['freq']],
        'sigreg':         [round(x, 6) for x in history['sigreg']],
    },
    'probe_linear':   probe_res['linear'],
    'probe_mlp':      probe_res['mlp'],
    'rollout_latent': rollout_lat,
    'rollout_pixel':  rollout_pix,
}

out_path = f'{VIS_DIR}/training_stats_rec.json'
Path(VIS_DIR).mkdir(exist_ok=True)
with open(out_path, 'w') as f:
    json.dump(stats, f, indent=2)

print(f"\n{'═'*58}")
print(f"  Stats → {out_path}")
print(f"  Temps    : {_total_time_s/60:.1f} min  |  {n_grad_steps:,} gradient steps")
print(f"  Énergie  : {energy_Wh:.3f} Wh  ({avg_power_W*_total_time_s:.0f} J)")
print(f"  Mémoire  : {peak_mem_MB:.0f} MB pic")
print(f"  Probe    : linear R²={probe_res['linear']['r2_mean']:.4f}  "
      f"mlp R²={probe_res['mlp']['r2_mean']:.4f}")
print(f"{'═'*58}")

## 8. Download checkpoint

In [ ]:
from google.colab import files
files.download(f'{CKPT_DIR}/lewm_rec_best.pt')